# Mecánica y paisajes de energía

Explora potenciales periódicos, coordenadas conformacionales, barreras energéticas, superficies 2D/3D y el efecto de la carga externa.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Resultados de aprendizaje

Al finalizar podrás interpretar un potencial bidimensional $U(x,q)$, leer mapas de contorno y superficies 3D, comparar perfiles a conformación fija, observar cómo una fuerza inclina un potencial periódico y distinguir un diagrama de energías asignadas manualmente de un paisaje termodinámico calibrado.

**Advertencia científica:** todas las superficies de este notebook son fenomenológicas y configurables. No se han obtenido de una estructura atómica, de dinámica molecular, de datos calorimétricos ni de las tasas de Gillespie.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Qué representa el paisaje

Un paisaje `U(x, q)` representa una energía potencial ilustrativa en función de una coordenada axial `x` (nm) y una coordenada conformacional adimensional `q` entre 0 y 1. Los resultados se expresan en unidades de `k_B T`. Los parámetros son fenomenológicos: no se deducen de las tasas de Gillespie ni son una superficie de energía libre experimental de kinesina.

### Coordenadas y función de energía

La clase `EnergyLandscape` define un potencial efectivo de dos coordenadas:

- $x$: posición axial sobre la pista, en nm.
- $q$: coordenada conformacional adimensional, restringida a $[0,1]$.
- $U/(k_BT)$: energía expresada en unidades de energía térmica.

La función implementada es:

$$
\frac{U(x,q)}{k_BT}=
U_{\mathrm{spatial}}+U_{\mathrm{conformational}}+
U_{\mathrm{bias}}+U_{\mathrm{coupling}}+U_{\mathrm{load}}.
$$

Sus términos son:

$$
U_{\mathrm{spatial}}=\frac{A_x}{2}\left[1-\cos\left(2\pi\left(\frac{x}{d}-\phi q\right)\right)\right],
$$

$$
U_{\mathrm{conformational}}=16A_q q^2(1-q)^2,
\qquad U_{\mathrm{bias}}=\Delta g\,q,
$$

$$
U_{\mathrm{coupling}}=J\sin\left(\frac{2\pi x}{d}\right)q(1-q),
\qquad U_{\mathrm{load}}=\frac{F x}{k_BT}.
$$

En la implementación, la última expresión se evalúa usando $F$ en pN, $x$ en nm y $k_BT$ en pN·nm. El signo positivo de $F$ representa una carga que se opone al movimiento en $+x$.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import numpy as np
import matplotlib.pyplot as plt

from protein_motor_sim.mechanics.energy_landscape import EnergyLandscape
from protein_motor_sim.visualization.energy_maps import (
    plot_energy_landscape_2d, plot_energy_landscape_3d, plot_energy_profiles
)
from protein_motor_sim.visualization.energy_landscape import (
    plot_periodic_energy_landscape, plot_conformational_energy_surface,
    plot_state_energy_diagram, thermal_energy_pN_nm
)

## 2. Construir un paisaje bidimensional

La superficie combina: periodicidad axial, doble pozo conformacional, sesgo, acoplamiento y una inclinación por carga externa. La carga opuesta añade un término lineal proporcional a `F x / (k_B T)`.

In [ ]:
landscape = EnergyLandscape(
    period_nm=8.2,
    spatial_barrier_kBT=2.5,
    conformational_barrier_kBT=3.0,
    conformational_bias_kBT=0.0,
    phase_shift=0.5,
    coupling_kBT=0.8,
    opposing_force_pN=0.0,
    temperature_K=310.0,
)
print("kBT a 310 K:", thermal_energy_pN_nm(310.0), "pN nm")

### Significado de los parámetros

| Parámetro | Significado | Unidad o dominio |
|---|---|---|
| `period_nm` ($d$) | Período espacial del potencial | nm |
| `spatial_barrier_kBT` ($A_x$) | Amplitud del término periódico | $k_BT$ |
| `conformational_barrier_kBT` ($A_q$) | Altura del doble pozo conformacional en el término aislado | $k_BT$ |
| `conformational_bias_kBT` ($\Delta g$) | Sesgo energético entre conformaciones | $k_BT$ |
| `phase_shift` ($\phi$) | Fracción del período desplazada al cambiar $q$ | fracción entre 0 y 1 |
| `coupling_kBT` ($J$) | Acoplamiento entre coordenadas | $k_BT$ |
| `opposing_force_pN` ($F$) | Carga opuesta al avance positivo | pN |

La coordenada $q$ es abstracta; no es automáticamente el grado real de docking de un neck linker. En el término conformacional aislado, $q=0$ y $q=1$ son mínimos y $q=1/2$ es el máximo. En la superficie total, los otros términos pueden modificar la ubicación y energía de los mínimos.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_energy_landscape_2d(landscape, x_range_nm=(-8.2, 32.8), ax=ax)
plt.show()

### Cómo leer el mapa 2D

Cada punto del plano corresponde a una pareja $(x,q)$ y el color representa $U/(k_BT)$. Las regiones de menor energía son valles del potencial elegido; las regiones de mayor energía forman crestas o barreras.

El mapa no muestra una trayectoria ni una probabilidad directamente. Para obtener una distribución de probabilidad habría que especificar el conjunto estadístico apropiado y, para una energía libre, considerar los grados de libertad que se han eliminado.

## 3. Superficie tridimensional

La tercera dimensión es la energía. Esta vista ayuda a reconocer valles, barreras y regiones conformacionales favorecidas.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
plot_energy_landscape_3d(landscape, x_range_nm=(-8.2, 32.8), ax=ax)
plt.show()

### Cómo leer la superficie 3D

La altura de la superficie es la energía. Los valles muestran combinaciones de posición y conformación de menor energía dentro de esta función; las crestas muestran combinaciones más costosas. La proyección 2D y la superficie 3D representan los mismos datos con distintas formas de visualización.

## 4. Perfiles a conformación fija

Al fijar `q`, observamos cómo cambia el potencial a lo largo de `x`. Se puede comparar el estado conformacional inicial, intermedio y final.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_energy_profiles(
    landscape,
    x_range_nm=(-8.2, 32.8),
    conformations={"q inicial": 0.0, "intermedio": 0.5, "q final": 1.0},
    ax=ax,
)
plt.show()

### Perfiles a conformación fija

La gráfica compara cortes de la superficie:

$$U(x,0),\qquad U(x,0.5),\qquad U(x,1).$$

En cada curva se mantiene fijo $q$ y se cambia $x$. Si los mínimos se desplazan entre perfiles, significa que el potencial espacial depende de la conformación mediante el término de fase $\phi q$ y el acoplamiento. La curva intermedia también incorpora la penalización del término de doble pozo.

Estos cortes ayudan a visualizar un acoplamiento posición–conformación, pero no prueban por sí mismos que la kinesina real siga esa ruta.

## 5. Cómo inclina el paisaje la carga externa

Compara potenciales periódicos sometidos a distintas fuerzas opuestas. Una carga positiva hace crecer la energía hacia `+x` en la convención de este módulo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_periodic_energy_landscape(
    x_range_nm=(0, 32.8),
    lattice_spacing_nm=8.2,
    barrier_kbt=4.0,
    forces_pN=(0.0, 1.0, 2.0, 4.0),
    temperature_K=310.0,
    ax=ax,
)
plt.show()

### Efecto de la carga externa

Para el potencial periódico unidimensional se añade una inclinación lineal $Fx/(k_BT)$. Con la convención de este módulo, una fuerza positiva hace crecer la energía hacia $+x$, es decir, se opone al avance en esa dirección.

La fuerza derivada de un potencial espacial es $F_x=-\partial U/\partial x$. Esta relación permite interpretar la pendiente del potencial. No obstante, KineSIM no integra actualmente una ecuación de movimiento sobre este paisaje; la gráfica es una exploración estática.

## Superficie conformacional alternativa: convención q en [-1, 1]

El paquete incluye otra función de visualización, `plot_conformational_energy_surface`, que utiliza una convención distinta para la coordenada conformacional: $q=-1$ se etiqueta como no acoplada (*undocked*) y $q=+1$ como acoplada (*docked*). Esta función construye un potencial fenomenológico con un doble pozo, un sesgo de docking y una barrera axial que depende de la conformación.

Esta coordenada **no es la misma** que la de `EnergyLandscape`, que está definida en $q\in[0,1]$. Se muestran como dos parametrizaciones ilustrativas independientes; no mezcles sus valores ni interpretes el signo de $q$ como una variable estructural medida.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax, contour = plot_conformational_energy_surface(
    x_range_nm=(-8.2, 32.8),
    lattice_spacing_nm=8.2,
    barrier_kbt=5.0,
    conformational_barrier_kbt=2.0,
    docking_bias_kbt=1.0,
    docking_barrier_coupling=0.35,
    opposing_force_pN=0.0,
    temperature_K=310.0,
    ax=ax,
)
fig.colorbar(contour, ax=ax, label=r"$U/(k_B T)$")
plt.show()


### Cómo interpretar esta superficie alternativa

El eje $q$ va de -1 a +1. Los extremos representan las etiquetas abstractas `undocked` y `docked`; la región central contiene una penalización conformacional. El parámetro `docking_bias_kbt` favorece una de las dos regiones y `docking_barrier_coupling` modifica la barrera espacial en función de $q$.

Al igual que la superficie principal, este mapa no está calibrado experimentalmente y no deriva las tasas de Gillespie. Su finalidad es explorar visualmente cómo un potencial elegido puede acoplar coordenadas espaciales y conformacionales.

## 6. Diagrama esquemático de estados y barreras

Este diagrama requiere introducir energías relativas y barreras de forma explícita. No es correcto inferirlas directamente de las tasas actuales, porque todavía no imponemos balance detallado local.

In [ ]:
state_energies = {
    "bound_apo": 0.0,
    "atp_bound": -1.0,
    "post_hydrolysis": -0.2,
}
barriers = {
    ("bound_apo", "atp_bound"): 2.5,
    ("atp_bound", "post_hydrolysis"): 2.2,
}
fig, ax = plt.subplots(figsize=(8, 4))
plot_state_energy_diagram(state_energies, barriers_kbt=barriers, ax=ax)
plt.show()

### Diagrama de energías de estados

Los valores de `state_energies` y `barriers` se introducen manualmente. Las barreras representan energías absolutas de estados de transición según la convención de la función de dibujo; no se infieren de las tasas de la red cinética.

Una red cinética irreversible o parcialmente especificada no determina por sí sola un perfil de energía libre único. Para relacionar energías y tasas hacen falta tasas reversibles, información sobre el trabajo químico del ATP/ADP/fosfato y restricciones de consistencia termodinámica.

## 7. Interpretar el resultado

- Un mínimo corresponde a una configuración relativamente estable en el potencial elegido.
- Una barrera alta dificulta una transición térmica.
- La carga inclina el paisaje y cambia el trabajo mecánico asociado al desplazamiento.
- El mapa es un modelo matemático de exploración, no un potencial de fuerza molecular ni una energía libre medida de kinesina.

### Límites e interpretación científica

1. $U(x,q)$ es una función fenomenológica, no un campo de fuerza atomístico ni una energía libre experimental de kinesina.
2. $q\in[0,1]$ en `EnergyLandscape`. Otras funciones de visualización del paquete emplean una convención independiente $q\in[-1,1]$; no deben mezclarse sin una transformación explícita.
3. El paisaje no se infiere de la red de Gillespie y no modifica sus tasas.
4. No se impone balance detallado local entre las rutas químicas y mecánicas.

## Referencias

- Kolomeisky, A. B., & Fisher, M. E. (2007). *Molecular motors: A theorist's perspective*. Annual Review of Physical Chemistry, 58, 675–695. [https://doi.org/10.1146/annurev.physchem.58.032806.104532](https://doi.org/10.1146/annurev.physchem.58.032806.104532)
- Ariga, T., Tomishige, M., & Mizuno, D. (2020). *Experimental and theoretical energetics of walking molecular motors under fluctuating environments*. Biophysical Reviews, 12, 503–510. [https://doi.org/10.1007/s12551-020-00684-7](https://doi.org/10.1007/s12551-020-00684-7)